# Consistency

Kafi Streams is *internally consistent* or you could also say *strongly consistent* thanks to the underlying pydbsp library.

This chapter is about a blog that changed the way I look at stream processing, and that was one of the main reasons why I started developing Kafi Streams at all.

It was the ingenious blog [Internal consistency in streaming systems](https://www.scattered-thoughts.net/writing/internal-consistency-in-streaming-systems/) by Jamie Brandon:

1. This blog first prompted me to co-author the book *Streaming Databases* (with Hubert Dulay) for O'Reilly (released in August 2024).
2. It then prompted me to write the blog [Why Streaming Still Isn't Mainstream](https://ralphmdebusmann.substack.com/p/why-streaming-still-isnt-mainstream) (August 2025).
3. And then to build my own stream processing library - Kafi Streams (first beta version 0.7.0: August 2026).

In this chapter, we summarize Jamie Brandon's blog in [Internal consistency in streaming systems](#internal).

Then, we set up a Kafi Streams topology modeling the example in [Topology](#topology) and [Test](#test) it.


## Overview

* [Internal consistency in streaming systems](#internal)
* [Topology](#topology)
* [Test](#test)


---
<a id="internal"></a>
## Internal consistency in streaming systems

Jamie Brandon's blog It showed how "classical" stream processors like Kafka Streams and Flink can easily be "tricked" by some simple lines of SQL and end up in emitting thousands of useless, and actually just *wrong* intermediate output messages.

At the time I read this blog for the first time (pretty late, early 2023), this was a revelation but I didn't yet know what to do about it. It seemed that everything that I had believed in about stream processing was off.

How does the example work?

The input is a stream of transactions moving money between 10 distinct bank accounts. The amount of money moved is always 1.

First we create two views for the credits and debits of each account:

```sql
CREATE VIEW credits AS
SELECT
    to_account AS account, 
    sum(amount) AS credits
FROM
    transactions
GROUP BY
    to_account;

CREATE VIEW debits AS
SELECT
    from_account AS account, 
    sum(amount) AS debits
FROM
    transactions
GROUP BY
    from_account;
```    

Second, we calculate their balances (= credits - debits):

```sql
CREATE VIEW balance AS
SELECT
    credits.account AS account, 
    credits.credits - debits.debits AS balance
FROM
    credits
INNER JOIN debits 
    ON credits.account = debits.account;
```

Since money is only being moved around, never created or destroyed, the sum of all the balances should always be 0. This is how we get the total sum:

```sql
CREATE VIEW total AS
SELECT
    sum(balance)
FROM
    balance;
```

The blog is from 2021. Today, without using extra machinery or workarounds, classical stream processors like Flink still badly fail at this example!

There is, however, a novel stream processor by Hartmut Armbruster that *can* correctly handle it: [StoatFlow](https://stoatflow.io/) (see also Hartmut's [blog](https://stoatflow.io/blog/internal-consistency) about the topic).

 

---
<a id="topology"></a>
## Topology

Here is a Kafi Streams topology modeling Jamie Brandon's example:

In [14]:
import sys

sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

#

transaction_source_str = "transactions"
sink_str = "total"

transaction_tn = (
    Tn.source(transaction_source_str)
    .map(lambda r: {"from_account": r["from_account"],
                    "to_account": r["to_account"],
                    "amount": r["amount"]})
)
#
credits_tn = (
    transaction_tn
    .group_by_sum(
        key_fun=lambda r: r["to_account"],
        value_fun=lambda r: r["amount"],
        project_fun=lambda key_int, value_int: {"account": key_int,
                                                "credits": value_int})
)
#
debits_tn = (
    transaction_tn
    .group_by_sum(
        key_fun=lambda r: r["from_account"],
        value_fun=lambda r: r["amount"],
        project_fun=lambda key_int, value_int: {"account": key_int,
                                                "debits": value_int})
)
#
balance_tn = (
    credits_tn
    .join_equi(debits_tn,
               left_key_fun=lambda l_r: l_r["account"],
               right_key_fun=lambda r_r: r_r["account"],
               project_fun=lambda l_r, r_r: {"account": l_r["account"],
                                             "balance": l_r["credits"] - r_r["debits"]})
)
#
sink_tn = (
    balance_tn.sum(value_fun=lambda x: x["balance"],
                   project_fun=lambda x: {"total": x})
).sink(sink_str)
#
_ = built_tn = Tn.build(sink_tn)


<a id="test"></a>
## Test

Now let's throw some data at the topology to test it:

In [ ]:
import random

def gen(n_int):
    r_list = []
    for _ in range(n_int):
        r = {"from_account": random.randint(0, 9),
             "to_account": random.randint(0, 9),
             "amount": 1}
        #
        r_list.append(r)
    #
    return r_list

batch_int = 100
steps_int = 100

collected_m_list = []
for step_int in range(0, batch_int):
    m_list = built_tn.process({transaction_source_str: gen(100)}).get(sink_str, [])
    #
    print(f"\rStep {step_int + 1} ({(step_int + 1) * batch_int}/{batch_int * steps_int}): {m_list}")
    


Step 1 (100/10000): [{'total': 0}]
Step 2 (200/10000): []
Step 3 (300/10000): []
Step 4 (400/10000): []
Step 5 (500/10000): []
Step 6 (600/10000): []
Step 7 (700/10000): []
Step 8 (800/10000): []
Step 9 (900/10000): []
Step 10 (1000/10000): []
Step 11 (1100/10000): []
Step 12 (1200/10000): []
Step 13 (1300/10000): []
Step 14 (1400/10000): []
Step 15 (1500/10000): []
Step 16 (1600/10000): []
Step 17 (1700/10000): []
Step 18 (1800/10000): []
Step 19 (1900/10000): []
Step 20 (2000/10000): []
Step 21 (2100/10000): []
Step 22 (2200/10000): []
Step 23 (2300/10000): []
Step 24 (2400/10000): []
Step 25 (2500/10000): []
Step 26 (2600/10000): []
Step 27 (2700/10000): []
Step 28 (2800/10000): []
Step 29 (2900/10000): []
Step 30 (3000/10000): []
Step 31 (3100/10000): []
Step 32 (3200/10000): []
Step 33 (3300/10000): []
Step 34 (3400/10000): []
Step 35 (3500/10000): []
Step 36 (3600/10000): []
Step 37 (3700/10000): []
Step 38 (3800/10000): []
Step 39 (3900/10000): []
Step 40 (4000/10000): []
Step 4

As you can see, after the first `100` input records have been processed, Kafi Streams correctly returns:
```
{'total': 0}
```
And as this `total` doesn't change, however many further messages we throw at the topology, all the other outputs are empty.
